# Generate reports for disaster monitoring

This pipeline creates disaster reports via LLMs, following the methods introduced by [Cantini et al. (2025)](https://www.sciencedirect.com/science/article/pii/S246869642400020X).

Prior to running this notebook, generate corresponding disaster reports via the method using the command (or similar).

```sh
python src/feedback_forensics/tools/external/disaster_report_writer.py --input ./data/input/humaid/hurricane_harvey_2017/hurricane_harvey_2017_train.tsv --model openrouter/openai/gpt-4o-mini-2024-07-18
```

If you want to reuse labels, adjust the command as below:

```sh
python src/feedback_forensics/tools/external/disaster_report_writer.py --input ./exp/disaster_report_writer/gemini_2.5_flash/tweets.tsv  --model openrouter/openai/gpt-4o-mini-2024-07-18 --reuse-labels
```





In [ ]:
import pandas as pd
import pathlib

def read_tsv(filepath):
    """Read a TSV file and return its contents."""
    import pandas as pd
    return pd.read_csv(filepath, sep='\t')


def create_report(row):
    """Create a report for a given row."""
    report = "title: " + row["title"] + "\n\n" + "introduction: " + row["introduction"] + "\n\n" + "content: " + row["content"]
    return report


save_path = pathlib.Path("../exp/disaster_report_writer")

paths = {
    "gemini_2.5_flash": save_path / "gemini_2.5_flash" / "reports.tsv",
    "gpt_4o_mini_2024_07_18": save_path / "gpt_4o_mini_2024_07_18" / "reports.tsv",
}

data = {}

for model, path in paths.items():
    df = read_tsv(path)
    df["report"] = df.apply(create_report, axis=1)
    data[model] = {"df": df, "path": path}
    print(f"Loaded {model} with {len(df)} rows")

In [ ]:
# Example report
print(data["gpt_4o_mini_2024_07_18"]["df"].iloc[10]["report"])

In [ ]:
# Merge all dataframes on city and state columns
# Start with the first dataframe
model_names = list(data.keys())
merged_df = data[model_names[0]]["df"].copy()

# rename columns to include model name (except city and state columns)
cols_to_rename = [col for col in merged_df.columns if col not in ["city", "state"]]
merged_df = merged_df.rename(columns={col: f"{col}_{model_names[0]}" for col in cols_to_rename})



# Merge with remaining dataframes
for model in model_names[1:]:
    df = data[model]["df"].copy()

    # Rename columns to include model name (except city and state columns)
    cols_to_rename = [col for col in df.columns if col not in ["city", "state"]]
    df = df.rename(columns={col: f"{col}_{model}" for col in cols_to_rename})

    # Merge on city and state and track dropped rows
    before_merge = len(merged_df)
    merged_df = merged_df.merge(df, on=["city", "state"], how="inner", suffixes=("", f"_{model}"))
    after_merge = len(merged_df)

    if before_merge != after_merge:
        print(f"Warning: {before_merge - after_merge} rows dropped when merging with {model}")
        # Find which city/state combinations were dropped
        merged_locations = set(zip(merged_df["city"], merged_df["state"]))
        df_locations = set(zip(df["city"], df["state"]))
        dropped_locations = merged_locations - df_locations
        if dropped_locations:
            print(f"  Dropped city/state combinations: {dropped_locations}")



print(f"Merged dataframe shape: {merged_df.shape}")
print(f"len(merged_df): {len(merged_df)}")
print(f"Columns: {merged_df.columns.tolist()}")
merged_df.head()


In [ ]:
df_to_save = merged_df.copy()
df_to_save["model_a"] = "gemini_2.5_flash"
df_to_save["model_b"] = "gpt_4o_mini_2024_07_18"
df_to_save["text_a"] = df_to_save["report_gemini_2.5_flash"]
df_to_save["text_b"] = df_to_save["report_gpt_4o_mini_2024_07_18"]

# remove other columns
df_to_save = df_to_save[["model_a", "model_b", "text_a", "text_b"]]

# save to csv
df_to_save.to_csv(save_path / "pairwise_reports_fully_separate_pipeline.csv", index=False)

Then run ICAI on this pipeline:

```
icai-exp -cd data/configs/004_env_reports/config.yaml
```

And then use Feedback Forensics to compute metrics:

In [ ]:
# new model analysis
import pandas as pd
import feedback_forensics as ff
import pathlib
import copy
import feedback_forensics.app.plotting.paper as paper_plot

fig_save_path = pathlib.Path("./output/png")
tex_save_path = pathlib.Path("./output/tex")
tex_app_save_path = pathlib.Path("./output/tex/appendix")

# ensure save path exists
fig_save_path.mkdir(parents=True, exist_ok=True)
tex_save_path.mkdir(parents=True, exist_ok=True)
tex_app_save_path.mkdir(parents=True, exist_ok=True)

# "../data/paper/model_comparison_annotated_ap.json"

def create_paper_plots(ap_data_path: str, appendix_str: str = ""):
    cache = {}
    dataset = ff.DatasetHandler(cache=cache)
    dataset.add_data_from_path(pathlib.Path(ap_data_path))
    annotator_metadata = dataset.get_available_annotators()

    models =[
        "gemini_2.5_flash",
        "gpt_4o_mini_2024_07_18",
    ]

    special_annotators = {
        annotator_key: metadata
        for annotator_key, metadata in annotator_metadata.items()
        if metadata.get("model_id") in models
    }

    replace_dict = {
        "gpt-4o-mini-2024-07-18": "GPT-4o-Mini",
        "gemini-2.5-flash": "Gemini-2.5-Flash",
    }

    for annotater_key, metadata in special_annotators.items():
        print(metadata["annotator_visible_name"])
        metadata["annotator_visible_name"] = metadata["annotator_visible_name"].replace("Model: ", "").replace("_", "-")
        #metadata["annotator_visible_name"] = metadata["annotator_visible_name"].split("/")[0] + " \\textit{" + metadata["annotator_visible_name"].split("/")[1] + "}"
        for key, value in replace_dict.items():
            metadata["annotator_visible_name"] = metadata["annotator_visible_name"].replace(key, value)
        print(metadata["annotator_visible_name"])




    dataset.set_annotator_cols(annotator_keys=list(special_annotators.keys()))
    df = dataset.get_annotator_metrics_df(metric_name="strength", index_col_name="Generate a response that...")

    # rename the columns
    #df.rename(rename_dict, inplace=True, axis=1)

    def reorder_columns(df):
        # reorder the columns (experts, regular, gpt-4)
        return df[['Generate a response that...',
        'Google \\textit{Gemini-2.5-pro}',
        'Mistral \\textit{Medium-3.1}',
        'OpenAI \\textit{GPT-oss-20b}',
        'xAI \\textit{Grok-4}',
        'Anthropic \\textit{Claude-Sonnet-4}',
        'OpenAI \\textit{GPT-5}',
        'Max diff']]

    #df = reorder_columns(df)

    latex_str = paper_plot.get_latex_table_from_metrics_df(
        metrics_df=df.head(5),
        title="Personality traits exhibited by different models",
        first_col_width=0.25,
    )

    with open(tex_save_path / f"010_env_reports{appendix_str}_model_comparison.tex", "w", encoding="utf-8") as f:
        f.write(latex_str)


    for metric in ["strength", "relevance", "cohens_kappa"]:
        df = dataset.get_annotator_metrics_df(metric_name=metric, index_col_name="Generate a response that...")
        #df = reorder_columns(df)

        latex_str = paper_plot.get_latex_table_from_metrics_df(
            metrics_df=df,
            title="Personality traits exhibited by different models",
            first_col_width=0.25,
        )

        with open(tex_app_save_path / f"010_env_reports{appendix_str}_model_comparison_long_{metric}.tex", "w", encoding="utf-8") as f:
            f.write(latex_str)

create_paper_plots("../exp/outputs/2025-11-03_15-07-06_separate/results/070_annotations_train_ap.json", "_separate")
create_paper_plots("../exp/outputs/2025-11-03_14-05-04_same_tweet_annotations/results/070_annotations_train_ap.json", "_gemini_labels")